# 아이 키 성장 기록 × 2017 성장도표 비교 분석

## 분석 질문
1. 아이의 키는 또래 남아 중앙값과 비교해 cm 차이와 백분위 기준으로 각각 어떻게 변해 왔는가?
2. 아이의 키 백분위가 가장 크게 오른 시기는 언제인가?
3. 4계절(봄·여름·가을·겨울)에 따라 아이의 성장 속도에 차이가 있는가?
4. 최근 1년 평균 백분위가 유지된다면 18세 때 키는 얼마일까?

## 3단계. 데이터 로드 및 기본 정보 확인

로컬에서 실행하면 `data/` 폴더를, Colab에서 실행하면 GitHub 저장소의 파일을 읽는다.

In [ ]:
from pathlib import Path
import pandas as pd

if Path("data").exists():
    BASE = "data/"
else:
    BASE = "https://raw.githubusercontent.com/Hoyasiro/M1-1/main/data/"

child = pd.read_csv(BASE + "child_growth_anon.csv")
child.shape

### 3-1. 컬럼과 자료형

In [ ]:
child.info()

### 3-2. 앞부분 미리보기

In [ ]:
child.head(15)

### 3-3. 컬럼별 결측치 개수

In [ ]:
child.isna().sum()

## 4단계. 결측치·이상치 처리

### 4-1. 결측치 처리
- 모든 컬럼이 빈 5행: 원본 엑셀에서 보기 편하도록 끼워 넣은 구분용 빈 줄이므로 삭제
- 키만 빈 행: 원본은 육아 기록이라 키 측정이 그날의 주목적이 아니었음. 재지 않은 값을 만들지 않기 위해 키 분석에서는 제외(실측 66개만 사용)

In [ ]:
child = child.dropna(how="all")
height = child.dropna(subset=["키"]).copy()
print(child.shape, height.shape)

### 4-2. 시간 순서 확인 (`days`가 계속 커지는가?)

In [ ]:
height["days"].is_monotonic_increasing

### 4-3. 같은 날 중복 측정

In [ ]:
height[height.duplicated("days", keep=False)]

### 4-4. 이전 측정보다 키가 줄어든 경우

In [ ]:
height = height.sort_values("days")
height["키_변화"] = height["키"].diff()
height[height["키_변화"] < 0]

### 4-5. 키 감소 이상치 처리
- 기준: 이전 측정보다 0.8cm 넘게 줄어든 값은 삭제
- 이유: 0.5~0.6cm 감소는 측정 자세·시간대에 따른 일반적 오차 범위로 보고 유지(이동평균으로 완화 가능).
  1.2cm 감소는 그보다 훨씬 커서 남겨 두면 이동평균선까지 끌어내려 추세를 왜곡할 수 있음

In [ ]:
height = height[~(height["키_변화"] < -0.8)].copy()
height["키_변화"] = height["키"].diff()
height.shape

### 4-6. 짧은 기간에 크게 뛴 값
측정 간격이 제각각이므로 키 변화를 1년 기준 속도(cm/년)로 바꿔 비교한다.

In [ ]:
height["간격_일"] = height["days"].diff()
height["성장속도_cm_년"] = height["키_변화"] / height["간격_일"] * 365
height.sort_values("성장속도_cm_년", ascending=False).head(5)

### 4-7. 성장 속도 계산 방식 결정
4-6에서 1일 간격에 1.8cm(연 657cm)처럼 불가능한 속도가 나왔다. 키 값이 틀렸다기보다
**간격이 너무 짧아 측정 오차가 부풀려진 것**이다. 따라서 키 실측값은 모두 유지하고,
성장 속도는 개별 측정 사이가 아니라 **계절 단위로 묶어서** 계산한다(5단계).

## 5단계. 시계열 분석

### 5-1. 계절 평균끼리 비교한 성장 속도
1. 연속된 같은 계절의 측정을 하나의 묶음으로 만든다(같은 계절이라도 92일 넘게 떨어지면 다른 해로 보고 분리)
2. 묶음마다 평균 키와 평균 생후일수를 구한다
3. 바로 앞 계절 묶음과 비교해 성장 속도(cm/년)를 계산한다
4. 계절을 건너뛴 경우(간격 150일 초과)는 여러 계절이 섞이므로 제외한다

한계: 계산된 속도는 "앞 계절 중간 → 이번 계절 중간" 구간이므로 순수하게 한 계절만의 성장은 아니다.

In [ ]:
계절_이름 = {12: "겨울", 1: "겨울", 2: "겨울", 3: "봄", 4: "봄", 5: "봄",
            6: "여름", 7: "여름", 8: "여름", 9: "가을", 10: "가을", 11: "가을"}
height["계절"] = height["month"].map(계절_이름)

새_묶음 = (height["계절"] != height["계절"].shift()) | (height["간격_일"] > 92)
height["묶음"] = 새_묶음.cumsum()

seasons = height.groupby("묶음").agg(
    계절=("계절", "first"),
    측정수=("키", "size"),
    평균_days=("days", "mean"),
    평균_키=("키", "mean"),
)
seasons["간격_일"] = seasons["평균_days"].diff()
seasons["성장속도_cm_년"] = seasons["평균_키"].diff() / seasons["간격_일"] * 365
seasons.loc[seasons["간격_일"] > 150, "성장속도_cm_년"] = None
seasons.round(1)

### 5-2. 계절별 평균 성장 속도

In [ ]:
계절_순서 = ["봄", "여름", "가을", "겨울"]
seasons.groupby("계절")["성장속도_cm_년"].agg(["count", "mean", "std"]).reindex(계절_순서).round(1)